## Tool Search for Context Management

### Installing Utilities and Libraries

In [2]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [4]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Implement the Weather API Call

In [5]:
import requests

def get_current_weather(location: str):

    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Implement the Sum Function

In [6]:
def calculate_sum(a: int, b: int) -> str:
    return str(a + b)

### Use the Messages API

In [9]:
import json
from datetime import datetime

LOG_PATH = "request_response_log.txt"


def summarize_request_content(content):
    """Render a message's content (str or list of blocks/dicts) as readable lines."""
    if isinstance(content, str):
        return [f"      text: {content!r}"]

    lines = []
    for block in content:
        if isinstance(block, dict):
            btype = block.get("type")
            if btype == "tool_result":
                lines.append(
                    f"      [tool_result] tool_use_id={block.get('tool_use_id')} "
                    f"content={str(block.get('content'))[:300]!r}"
                )
            else:
                lines.append(f"      [{btype}] {str(block)[:300]}")
        else:
            btype = getattr(block, "type", "?")
            if btype == "text":
                lines.append(f"      [text] {block.text[:300]!r}")
            elif btype == "tool_use":
                lines.append(f"      [tool_use] name={block.name} id={block.id} input={json.dumps(block.input)}")
            elif btype == "mcp_tool_listing":
                tool_names = [t.name for t in getattr(block, "tools", [])]
                lines.append(
                    f"      [mcp_tool_listing] server={getattr(block, 'server_name', '?')} "
                    f"tool_count={len(tool_names)} tools={tool_names}"
                )
            else:
                lines.append(f"      [{btype}] {str(block)[:300]}")
    return lines


def log_request(f, call_number, messages, tools, mcp_servers):
    f.write(f"\n{'=' * 78}\n")
    f.write(f"CALL #{call_number} — REQUEST\n")
    f.write(f"{'=' * 78}\n")
    f.write(f"model: {claude_model_name}\n")
    f.write(f"tools sent ({len(tools)}): {[t.get('name') for t in tools]}\n")
    f.write(f"mcp_servers sent: {[s.get('name') for s in mcp_servers]}\n")
    f.write(f"messages sent ({len(messages)} total):\n")
    for i, m in enumerate(messages):
        f.write(f"  [{i}] role={m['role']}\n")
        for line in summarize_request_content(m["content"]):
            f.write(line + "\n")
    f.flush()


def log_response(f, call_number, response):
    usage = response.usage
    f.write(f"\n{'-' * 78}\n")
    f.write(f"CALL #{call_number} — RESPONSE\n")
    f.write(f"{'-' * 78}\n")
    f.write(f"stop_reason: {response.stop_reason}\n")
    f.write("usage:\n")
    f.write(f"  input_tokens              (billed full price) = {usage.input_tokens}\n")
    f.write(f"  cache_creation_input_tokens (cache write)      = {getattr(usage, 'cache_creation_input_tokens', 0)}\n")
    f.write(f"  cache_read_input_tokens     (cache hit)        = {getattr(usage, 'cache_read_input_tokens', 0)}\n")
    f.write(f"  output_tokens                                  = {usage.output_tokens}\n")
    total_input = (
        usage.input_tokens
        + (getattr(usage, "cache_creation_input_tokens", 0) or 0)
        + (getattr(usage, "cache_read_input_tokens", 0) or 0)
    )
    f.write(f"  => total input (input + cache_write + cache_read) = {total_input}\n")
    f.write(f"content blocks returned ({len(response.content)}):\n")
    for line in summarize_request_content(response.content):
        f.write(line + "\n")
    f.flush()


messages = [
    {
        "role": "user",
        "content": "What is the weather in London like currently?"
    }
]

tools = [

    {
        "type": "tool_search_tool_bm25_20251119",
        "name": "tool_search_tool_bm25"
    },
    {
        "type": "mcp_toolset",
        "mcp_server_name": "Microsoft Learn",
        "default_config": {
            "enabled": True,
            "defer_loading": True
        }
    },
    {
        "name": "GetCurrentWeather",
        "description": "Retrieve the current weather information for a specified location using the wttr.in weather service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City or location to retrieve the weather for."
                }
            },
            "required": ["location"]
        },
        "defer_loading": True
    },
    {
        "name": "CalculateSum",
        "description": "Calculate the sum of two integer numbers and return the result.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "First integer."
                },
                "b": {
                    "type": "integer",
                    "description": "Second integer."
                }
            },
            "required": ["a", "b"]
        },
        "defer_loading": True
    }

]

mcp_servers = [
    {
        "type": "url",
        "url": "https://learn.microsoft.com/api/mcp",
        "name": "Microsoft Learn"
    }
]

call_number = 0

with open(LOG_PATH, "w") as log_file:
    log_file.write("Claude Messages API — Request/Response Log\n")
    log_file.write(f"Run started: {datetime.now().isoformat()}\n")

while True:

    call_number += 1

    with open(LOG_PATH, "a") as log_file:
        log_request(log_file, call_number, messages, tools, mcp_servers)

    response = client.beta.messages.create(

        model=claude_model_name,

        max_tokens=4096,

        messages=messages,

        mcp_servers=mcp_servers,

        tools=tools,

        betas=["mcp-client-2025-11-20"],
        
        cache_control={"type": "ephemeral"}

    )

    with open(LOG_PATH, "a") as log_file:
        log_response(log_file, call_number, response)


    # Add Claude's response to the conversation
    messages.append(
        {
            "role": "assistant",
            "content": response.content
        }
    )

    tool_used = False


    # Process every content block
    for block in response.content:

        if block.type == "text":

            print(block.text)

        elif block.type == "tool_use":

            tool_used = True

            print(f"\nInvoking Tool: {block.name}\n")

            # Local Weather Tool
            if block.name == "GetCurrentWeather":

                result = get_current_weather(
                    block.input["location"]
                )

            # Local Calculator Tool
            elif block.name == "CalculateSum":

                result = calculate_sum(
                    block.input["a"],
                    block.input["b"]
                )

            # Skip MCP tools
            else:
                print(f"MCP tools are executed server-side by Claude")

                # MCP tools are executed server-side by Claude
                continue


            # Return tool result
            messages.append(
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result)
                        }
                    ]
                }
            )

    # Exit if no local tool was invoked
    if not tool_used:
        break

print(f"\nFull request/response log written to: {LOG_PATH}")



Invoking Tool: GetCurrentWeather

Here's the current weather in London (as of 02:33 PM local time):

🌥️ **Overcast**, 22°C (71°F), feels like 18°C (64°F)
- **Cloud cover:** 98%
- **Humidity:** 55%
- **Wind:** 22 km/h (14 mph) from the SSW
- **Visibility:** 10 km (6 miles)
- **Pressure:** 1012 mb
- **UV Index:** 1 (low)
- **Precipitation:** None currently

So it's a mild, cloudy day with no rain at the moment — a light jacket should be enough given the breezy conditions.

Full request/response log written to: request_response_log.txt
